In [ ]:
import json
import math
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from tqdm import tqdm

# ===============================================================
# CONFIGURATION
# ===============================================================
DATA_PATH = "../data/CDG-FCO/flight_data_with_minutes_since_start.json"
TEST_SAMPLE_PATH = "../data/CDG-FCO/RESULTS/test_sample.json"
OUTPUT_PATH = "../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_ARIMA.json"

PHASES = ["take_off", "cruising", "landing"]

EARTH_RADIUS_KM = 6371.0088


# ===============================================================
# HAVERSINE (no external import)
# ===============================================================
def haversine_km(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return EARTH_RADIUS_KM * c


def mean_haversine_km(gt_positions, pred_positions):
    if not gt_positions or not pred_positions:
        return None

    n = min(len(gt_positions), len(pred_positions))
    if n == 0:
        return None

    distances = [
        haversine_km(
            gt_positions[i][0], gt_positions[i][1],
            pred_positions[i][0], pred_positions[i][1]
        )
        for i in range(n)
    ]

    return float(np.mean(distances)) if distances else None


# ===============================================================
# ARIMA HELPERS
# ===============================================================
def fit_arima_and_forecast(series, steps, order=(1, 1, 1)):
    if steps <= 0:
        return []

    if len(series) < 5:
        return [series[-1]] * steps

    try:
        model = ARIMA(series, order=order)
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=steps)
        return forecast.tolist()
    except Exception:
        return [series[-1]] * steps


# ===============================================================
# MAIN PIPELINE
# ===============================================================
def run_arima_benchmark():
    # --------------------------------------------------
    # Load flights
    # --------------------------------------------------
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        flights_data = json.load(f)

    flights_by_id = {
        flight.get("flight_metadata", {}).get("fr24_id"): flight
        for flight in flights_data
    }

    # --------------------------------------------------
    # Load phase-specific samples
    # --------------------------------------------------
    with open(TEST_SAMPLE_PATH, "r", encoding="utf-8") as f:
        test_samples = json.load(f)

    phase_errors = {phase: [] for phase in PHASES}

    # --------------------------------------------------
    # ARIMA per phase
    # --------------------------------------------------
    for phase in PHASES:
        print(f"\nRunning ARIMA for phase: {phase}")

        for flight_id in tqdm(test_samples[phase], desc=f"{phase} flights"):
            flight = flights_by_id.get(flight_id)
            if flight is None:
                continue

            phase_data = flight.get(phase, [])
            if not phase_data:
                continue

            # ----------------------------------------------
            # Same spoofing logic as other pipelines
            # ----------------------------------------------
            N = len(phase_data)
            SPOOF_INDEX = N // 2

            observed_data = phase_data[:SPOOF_INDEX]
            spoofed_data = phase_data[SPOOF_INDEX:]

            if not observed_data or not spoofed_data:
                continue

            # Observed
            lats_obs = [p["lat"] for p in observed_data]
            lons_obs = [p["lon"] for p in observed_data]

            # Ground truth
            lats_gt = [p["lat"] for p in spoofed_data]
            lons_gt = [p["lon"] for p in spoofed_data]

            n_future = len(spoofed_data)

            # ----------------------------------------------
            # ARIMA prediction
            # ----------------------------------------------
            pred_lats = fit_arima_and_forecast(lats_obs, n_future)
            pred_lons = fit_arima_and_forecast(lons_obs, n_future)

            pred_positions = list(zip(pred_lats, pred_lons))
            gt_positions = list(zip(lats_gt, lons_gt))

            err_km = mean_haversine_km(gt_positions, pred_positions)
            if err_km is None:
                continue

            phase_errors[phase].append(err_km)

    return phase_errors


# ===============================================================
# SAVE RESULTS
# ===============================================================
def save_results(phase_errors):
    payload = {
        "ARIMA": phase_errors
    }

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4)

    print(f"\nSaved ARIMA results to {OUTPUT_PATH} ✅")



In [ ]:
phase_errors = run_arima_benchmark()
save_results(phase_errors)